## 1. Configuration and Setup

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import PyTorchModelHubMixin

MODEL_PATH = "amirbhat/motivbot-multilingual-XLMR-10062026v3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN = 256

COLS = [
    'Anxiety_Fear_Worry',
    'Depression_Hopelessness_Sadness',
    'Self_Esteem_Issue',
    'Positive_Affect',
    'Negative_Affect'
]

TERNARY_NAMES = ['No', 'Low', 'High']

# 2. Model Architecture Definition

In [ ]:
class MentalHealthClassifier(nn.Module, PyTorchModelHubMixin):
    def __init__(self, config=None):
        super().__init__()
        self.xlmr  = AutoModel.from_pretrained('xlm-roberta-base')
        self.drop  = nn.Dropout(p=0.3)
        self.fusion = nn.Sequential(
            nn.Linear(768 + 4, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.heads_3 = nn.ModuleList([nn.Linear(512, 3) for _ in range(5)])

    def forward(self, ids, mask, struct):
        out   = self.xlmr(input_ids=ids, attention_mask=mask)
        cls   = out.last_hidden_state[:, 0, :]
        fused = torch.cat([cls, struct], dim=1)
        fused = self.fusion(self.drop(fused))
        return [h(fused) for h in self.heads_3]

# 3. Load Model and Tokenizer

In [ ]:
print(f"Loading calibrated Multitask Fusion Engine from: {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = MentalHealthClassifier.from_pretrained(MODEL_PATH).to(device)
model.eval()

Loading calibrated Multitask Fusion Engine from: amirbhat/motivbot-multilingual-XLMR-10062026v3...


tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

MentalHealthClassifier(
  (xlmr): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
    

# 4. Inference Function




In [ ]:
# def predict_text_only(sentences, temperature=0.45):
#     """
#     Inference pipeline: Imputes neutral baselines for structural layers
#     """
#     all_results = []

#     for text in sentences:
#         # Impute population mean (Z-Score = 0.0) to safely bypass manual clinical metadata inputs
#         neutral_metadata = np.zeros(4, dtype=np.float32)
#         struct_t = torch.tensor(neutral_metadata).unsqueeze(0).to(device)

#         inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding='max_length').to(device)

#         with torch.no_grad():
#             outputs = model(inputs['input_ids'], inputs['attention_mask'], struct_t)

#         row_data = {"Text": text}

#         for idx, col_display in enumerate(COLS):
#             # Apply temperature scaling calibration
#             scaled_logits = outputs[idx] / temperature
#             probs = F.softmax(scaled_logits, dim=1).squeeze(0).cpu().numpy()

#             p_no, p_low, p_high = probs[0], probs[1], probs[2]

#             # 1. Paradigm A: Raw Ternary Decoding (No / Low / High)
#             pred_3class_idx = probs.argmax()
#             pred_3class_label = TERNARY_NAMES[pred_3class_idx]
#             p_ternary_confidence = probs[pred_3class_idx] * 100

#             # 2. Paradigm B: Argmax-Aligned Binary Flagging
#             if pred_3class_label == "High":
#                 binary_label = "YES"
#                 p_binary_confidence = p_high * 100
#             else:
#                 binary_label = "No"
#                 p_binary_confidence = (p_no + p_low) * 100

#             # Store computed metrics back to presentation mapping targets
#             row_data[f"{col_display}_3Class"] = f"{pred_3class_label} ({p_ternary_confidence:.1f}%)"
#             row_data[f"{col_display}_Binary"] = f"{binary_label} ({p_binary_confidence:.1f}%)"

#         all_results.append(row_data)

#     # Establish visual presentation formatting boundaries for clear matrix inspection
#     presentation_order = ['Text']
#     for task in COLS:
#         presentation_order.append(f"{task}_3Class")
#         presentation_order.append(f"{task}_Binary")

#     return pd.DataFrame(all_results)[presentation_order]


def predict_with_metadata(sentences, ages, sexes, rses_values, phq_values, temperature=0.45):
    """
    Inference pipeline: Accepts text + structured metadata (age, sex, RSES, PHQ).
    - sentences: list of strings
    - ages: list of floats/ints
    - sexes: list of floats/ints (e.g., 0=male, 1=female or your encoding)
    - rses_values: list of floats/ints
    - phq_values: list of floats/ints
    """
    all_results = []

    for text, age, sex, rses, phq in zip(sentences, ages, sexes, rses_values, phq_values):
        # Build structured feature vector
        metadata = np.array([age, sex, rses, phq], dtype=np.float32)
        struct_t = torch.tensor(metadata).unsqueeze(0).to(device)

        # Tokenize text
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=MAX_LEN, padding='max_length').to(device)

        with torch.no_grad():
            outputs = model(inputs['input_ids'], inputs['attention_mask'], struct_t)

        row_data = {"Text": text}

        for idx, col_display in enumerate(COLS):
            scaled_logits = outputs[idx] / temperature
            probs = F.softmax(scaled_logits, dim=1).squeeze(0).cpu().numpy()

            pred_3class_idx = probs.argmax()
            pred_3class_label = TERNARY_NAMES[pred_3class_idx]
            p_ternary_confidence = probs[pred_3class_idx] * 100

            if pred_3class_label == "High":
                binary_label = "YES"
                p_binary_confidence = probs[2] * 100
            else:
                binary_label = "No"
                p_binary_confidence = (probs[0] + probs[1]) * 100

            row_data[f"{col_display}_3Class"] = f"{pred_3class_label} ({p_ternary_confidence:.1f}%)"
            row_data[f"{col_display}_Binary"] = f"{binary_label} ({p_binary_confidence:.1f}%)"

        all_results.append(row_data)

    presentation_order = ['Text']
    for task in COLS:
        presentation_order.append(f"{task}_3Class")
        presentation_order.append(f"{task}_Binary")

    return pd.DataFrame(all_results)[presentation_order]


# 5. Execute Tests

In [ ]:
sentences = ["Herkes beni sinirlendiriyor, artık tahammülüm kalmadı"]
ages = [25]
sexes = [0]   # e.g., 0=male, 1=female
rses_values = [18]  # Rosenberg Self-Esteem Scale score
phq_values = [12]   # PHQ-9 score

results_df = predict_with_metadata(sentences, ages, sexes, rses_values, phq_values)
results_df

,Text,Anxiety_Fear_Worry_3Class,Anxiety_Fear_Worry_Binary,Depression_Hopelessness_Sadness_3Class,Depression_Hopelessness_Sadness_Binary,Self_Esteem_Issue_3Class,Self_Esteem_Issue_Binary,Positive_Affect_3Class,Positive_Affect_Binary,Negative_Affect_3Class,Negative_Affect_Binary
0,"Herkes beni sinirlendiriyor, artık tahammülüm ...",High (88.6%),YES (88.6%),No (84.5%),No (89.0%),No (93.6%),No (99.4%),No (48.3%),No (91.1%),High (38.5%),YES (38.5%)


In [ ]:
# if __name__ == "__main__":
#     production_test_sentences = [
#         "قلبم از نگرانی به شدت می‌تپد و خیلی استرس دارم که چه اتفاقی می‌افتد",
#         "Gelecek hakkında çok endişeliyim, her an kötü bir şey olacakmış gibi hissediyorum",
#         "دیگر هیچ امیدی به زندگی ندارم و همه‌اش می‌خواهم بخوابم و بیدار نشوم",
#         "Hayatın hiçbir anlamı kalmadı, kendimi derin bir boşlukta hissediyorum",
#         "امروز زیباترین روز زندگی من است و از ته دل خوشحالم",
#         "Çok mutluyum, her şey yolunda gidiyor ve çok enerjim var",
#         "من به توانایی‌های خودم ایمان دارم و می‌دانم که در این کار موفق می‌شوم",
#         "Kendime güveniyorum ve zorlukların üstesinden gelebileceğimi biliyorum",
#         "از همه چیز متنفرم و همه دارند روی اعصاب من راه می‌روند",
#         "Herkes beni sinirlendiriyor, artık tahammülüm kalmadı"
#     ]

#     results_df = predict_text_only(production_test_sentences, temperature=0.45)

#     pd.set_option('display.max_colwidth', 22)
#     pd.set_option('display.width', 1600)
#     pd.set_option('display.max_columns', None)






In [ ]:
results_df

,Text,Anxiety_Fear_Worry_3Class,Anxiety_Fear_Worry_Binary,Depression_Hopelessness_Sadness_3Class,Depression_Hopelessness_Sadness_Binary,Self_Esteem_Issue_3Class,Self_Esteem_Issue_Binary,Positive_Affect_3Class,Positive_Affect_Binary,Negative_Affect_3Class,Negative_Affect_Binary
0,"Herkes beni sinirlendiriyor, artık tahammülüm ...",High (88.6%),YES (88.6%),No (84.5%),No (89.0%),No (93.6%),No (99.4%),No (48.3%),No (91.1%),High (38.5%),YES (38.5%)
